# Citadel JWT Authentication & App Role Authorization - Testing Center

## Validate JWT-based Authentication and Role-Based Access Control Across Universal and Unified APIs

Use this Jupyter notebook to verify unified JWT authentication and app role authorization in Citadel Access Contracts, including:
- **Unified Security Handler**: Single `security-handler` fragment across Universal LLM and Unified AI APIs
- **API Key + JWT (enforced)**: Access contracts that require both API key and valid JWT Bearer token
- **App Role Authorization**: Access contracts that require specific app roles (e.g., `Model.Invoke`, `MCP.Invoke`, `Agent.Invoke`) in the JWT `roles` claim
- **Two API Surfaces**: Universal `/models/chat/completions` and Unified OpenAI v1 `/unified-ai/v1/chat/completions`, with the model in each request body
- **Negative Testing**: Verifying that requests without required JWT or app roles are rejected
- **Dual Auth Flows**: Automatic client-credentials tokens and a delegated token for the current Azure CLI user

> **Note:** This notebook assumes a fully deployed Citadel Governance Hub with JWT authentication configured.
> The gateway validates tokens using APIM named values (`JWT-OpenIdConfigUrl`, `JWT-Issuer`, `JWT-AppRegistrationId`)
> or per-product custom overrides (`jwtAudience`, `jwtIssuer`, `jwtOpenIdConfigUrl`) set in the product policy.
>
> **Setup references:**
> - [JWT Authentication Guide](../guides/entraid-auth-validation.md) - Gateway-side JWT configuration (named values & custom overrides)
> - [JWT Client Identity & Permissions Guide](../guides/jwt-client-identity-permissions.md) - Direct application role assignments and user-group assignments
> - [Entra ID Setup](../bicep/infra/entra-id-setup/) - Auto-provisioning Entra ID app registration with app roles
>
> **Entra app-role assignments:** Assign roles directly to application and managed-identity service principals. Security-group assignments supply roles to signed-in users, not service-principal members.
> App roles and exposed delegated scopes share a default limit of **700 permission definitions per application or service principal**, and also count toward the **1,200-entry application manifest limit**. These are definition limits, not assignment counts. See [Microsoft Entra app roles and limits](https://learn.microsoft.com/en-us/entra/identity-platform/howto-add-app-roles-in-apps#app-role-limits).

### Shared workflow and safety

Run from the repository root or `validation`. Explicit `overrides` win over the
selected azd environment. Annotate deployed model operations with `capabilities`
(or `VALIDATION_MODEL_CAPABILITIES`); unknown capabilities block required coverage.
Image operations use `image-generation` and `image-edit`; affinity uses `responses`.
Definitions and policy XML remain visible. Preview the generated **`.bicepparam`**
in one cell, optionally edit it, then deploy it in the following cell; the shared
deploy helper compiles the actual parameter file before submission.
Generated artifacts live under ignored `validation/.runs`. Tokens/keys stay in
memory. `run.finish()` raises for required FAIL/BLOCKED/empty runs. Cleanup is off
and can only delete resources absent before this run. Cloud execution has real cost
and side effects; local source validation is not a cloud feature verdict.

<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

Run the updated Entra setup script successfully in the selected azd environment,
then Run All using the same Azure CLI user. No manual fields, secrets or JWTs
are required. Setup stores client credentials in the private local azd environment.

Leave init_from_azd=True for automatic configuration. environment, overrides,
model_override and the optional credential/token fixtures remain available.


### 0️⃣ Initialize Notebook Variables

Configure this notebook's environment, explicit feature opt-ins, and run-owned resources without deploying anything.

**Input:** The selected repository kernel; edit the configuration values below and provide required secrets through environment variables.

**Output:** Configuration globals for the following initialization step; no cloud deployment or test is performed.

Run All loads setup outputs and acquires tokens for the Model.Invoke client, current signed-in Azure CLI user, and MCP.Invoke-only client without Model.Invoke. No manual fields or JWTs are required. Credential/token fixtures are optional overrides. Sign-in/MFA requirements are reported explicitly.


### Automatic models and optional overrides

Leave `environment=None`, azd enabled, `operation_overrides=None`, and
`skipped_models=None` to use the selected deployment without editing model lists.
Operations default to name-based **assumptions**, verified by real requests:
`image`/`flux` → image generation/edit; otherwise `embedding`/`text` → embeddings;
otherwise `gpt` → chat and Responses; other names → chat. Matching ignores case.

Optional examples (replace example names with names in the loaded inventory):

```python
# Replace operations only for these models; other models retain automatic mapping.
operation_overrides = {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
# Use operation_overrides = {} to discard stored azd operation overrides.
# Disable one model's tests with an empty override: {"my-model": []}.
skipped_models = ["mini", "legacy"]  # Case-insensitive substrings; backend deployment is unchanged.
overrides = {"resource_group": "rg-other-hub", "apim_name": "apim-other-hub"}  # Explicit azd-value overrides.
# To require a complete manually mapped catalogue: overrides["infer_operations"] = False.
```

Streaming is assumed for inferred chat models. Explicit mappings need `streaming`
when that scenario is selected. Responses GET/input-items returning HTTP 404 are warnings/SKIP for every model,
including explicit overrides, not successful retrieval. Other unsupported-operation
exceptions remain limited to inferred GPT-mini models; explicit mappings require
those non-404 retrieval checks. Authentication, throttling, and server failures
are never silently accepted.

### Additional scenario inputs

- `entra_client_secret` optionally reads `VALIDATION_ENTRA_CLIENT_SECRET`; otherwise it loads setup's saved azd secret. Never save secrets in notebook source.
- `no_role_token_fixture` optionally reads `VALIDATION_JWT_NO_ROLE_TOKEN`; otherwise a fresh token is acquired for setup's MCP.Invoke-only negative client.
- `device_token_fixture` optionally reads `VALIDATION_JWT_DEVICE_TOKEN`; otherwise the current Azure CLI user's delegated token is acquired automatically.

### Scenario-specific input examples


Entra coordinates and secrets come from setup in the selected azd environment.
The variables above are optional overrides. Both positive identities are always
tested; token_source selects only active_token for inspection. A genuine token
without Model.Invoke is acquired automatically for the denial control.


In [1]:
from pathlib import Path
import sys
repo_root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "azure.yaml").is_file() and (p / "shared" / "utils.py").is_file()), None)
if repo_root is None:
    raise RuntimeError("Open this repository and run from its root or validation directory.")
sys.path.insert(0, str(repo_root / "shared"))
import utils
with utils.notebook_step('0️⃣ Initialize Notebook Variables', inputs={'repository': str(repo_root if 'repo_root' in globals() else root)}, expected='Configuration globals for the following initialization step; no cloud deployment or test is performed.', namespace=globals()):
    import os, sys, json, time, uuid, base64, requests
    from pathlib import Path
    from apimtools import APIMClientTool
    from validationtools import ValidationContext, ValidationRun, RequestTransport, ValidationBlocked
    from jwttools import jwt_configuration, acquire_application_token, acquire_user_token, signed_in_user, validate_jwt
    from contracttools import write_parameters, deploy_contract, access_policy, preview
    from scenariotools import access_parameters, contract_identity, cleanup_products, policy_xml, completion_text, save_manifest

    init_from_azd = True
    environment = None  # Explicit azd environment, or the active environment.
    overrides = {}  # Explicit values win: resource_group/location/subscription_id/apim_name.
    # Supply capability metadata, not name patterns, e.g.
    # overrides["capabilities"] = {"your-deployed-model": ["chat", "responses"]}
    run_id = uuid.uuid4().hex[:12]
    cleanup_delete_contract = False

    model_override = None
    inference_api_version = "2024-05-01-preview"
    entra_client_secret = os.environ.get("VALIDATION_ENTRA_CLIENT_SECRET", "")
    no_role_token_fixture = os.environ.get("VALIDATION_JWT_NO_ROLE_TOKEN", "")
    device_token_fixture = os.environ.get("VALIDATION_JWT_DEVICE_TOKEN", "")
    token_source = "client_credentials"
    use_keyvault_integration = False
    keyvault_subscription_id = ""
    keyvault_resource_group = ""
    keyvault_name = ""

    # None uses azd operation overrides when present, otherwise automatic name-based mapping.
    # Example: {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
    # An empty list for one model disables its tests; explicit lists replace that model's inferred operations.
    # A supplied dictionary replaces azd's operation map; {} resets it to automatic mapping.
    operation_overrides = None
    # None uses VALIDATION_SKIPPED_MODELS from azd (default []); matches substrings without case sensitivity.
    # Example: ["mini", "legacy"] skips any model containing either keyword in validation, not deployment.
    skipped_models = None


<a id='1'></a>
### 1️⃣ Verify Azure CLI and Connected Subscription

Ensure Azure CLI is authenticated and connected to the correct subscription:

### 1️⃣ Verify Azure CLI and Connected Subscription

Resolve the selected deployment and validate this scenario's mandatory inputs before deployment or feature tests.

**Input:** Configuration from the previous cell, resource coordinates, inferred or explicitly overridden model operations, and enabled integration prerequisites.

**Output:** Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.


In [2]:
with utils.notebook_step('1️⃣ Verify Azure CLI and Connected Subscription', inputs={'init_from_azd': init_from_azd}, expected='Setup inputs resolved automatically; manual overrides remain available.', namespace=globals()):
    run = ValidationRun("jwt-authentication")
    context = ValidationContext.load(overrides=overrides, environment=environment, from_azd=init_from_azd,
        operation_overrides=operation_overrides, skipped_models=skipped_models)
    jwt_settings = jwt_configuration(context.config, context.account,
        client_secret=entra_client_secret, no_role_token=no_role_token_fixture,
        user_token=device_token_fixture)
    entra_tenant_id = jwt_settings["tenant_id"]
    entra_client_id = jwt_settings["client_id"]
    entra_audience = jwt_settings["audience"]
    entra_client_secret = jwt_settings["client_secret"]
    no_role_token_fixture = jwt_settings["no_role_token"]
    device_token_fixture = jwt_settings["user_token"]
    if token_source not in ("client_credentials", "current_user", "device_flow"):
        raise ValueError("token_source must be client_credentials or current_user (device_flow is a compatibility alias).")
    governance_hub_resource_group = context.config["resource_group"]
    location = context.config["location"]
    subscription_id = context.config["subscription_id"]
    llm_backends_config = context.config["llm_backends"]
    utils.require_capabilities(context.catalog, context.catalog.models_for("chat"))
    try:
        model_name = context.catalog.select('chat', override=model_override)
    except ValidationBlocked as exc:
        run.blocked("model-preflight", str(exc))
        raise
    keyvault_subscription_id = keyvault_subscription_id or subscription_id
    keyvault_resource_group = keyvault_resource_group or governance_hub_resource_group
    keyvault_name = keyvault_name or context.config.get("key_vault_name", "")
    if use_keyvault_integration:
        utils.require_inputs({"keyvault.name": keyvault_name})
    run_dir = repo_root / "validation" / ".runs" / "jwt-authentication" / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    transport = RequestTransport(timeout=120)
    utils.format_output('Resolved JWT validation target', {
        "environment": environment or "active", "resource_group": governance_hub_resource_group,
        "apim": context.config["apim_name"], "tenant": entra_tenant_id,
        "resource_client_id": jwt_settings["resource_id"], "model": model_name,
        "identities": ["service principal", "current Azure CLI user", "client without Model.Invoke"],
    })


### 2️⃣ Initialize APIM Client Tool

Prepare the 2️⃣ Initialize APIM Client Tool state shown below for its subsequent feature-specific requests and assertions.

**Input:** `governance_hub_resource_group`, `model_name`, `inference_api_version`

**Output:** `apimClientTool`, `universal_api`, `unified_api`, `universal_llm_chat_url`, and `unified_ai_v1_chat_url` are available to the following cells. No legacy OpenAI API is required.


In [3]:
with utils.notebook_step('2️⃣ Initialize APIM Client Tool', inputs={'governance_hub_resource_group': globals().get('governance_hub_resource_group'), 'model_name': globals().get('model_name'), 'inference_api_version': globals().get('inference_api_version')}, expected='Discover only Universal and Unified APIs; prepare Universal and Unified OpenAI v1 chat URLs.', namespace=globals()):
    apimClientTool = APIMClientTool(governance_hub_resource_group, apim_resource_name=context.config["apim_name"])
    apimClientTool.initialize_context(context.account, enumerate_subscriptions=False)
    universal_api = apimClientTool.discover_api_exact("models")
    unified_api = apimClientTool.discover_api_exact("unified-ai")
    gateway_url = str(apimClientTool.apim_resource_gateway_url).rstrip("/")
    azure_endpoint = gateway_url + "/"
    apim_resource_gateway_url = gateway_url
    apim_name = apimClientTool.apim_resource_name
    preexisting_products = {p.name for p in apimClientTool.client.product.list_by_service(
        governance_hub_resource_group, apim_name)}
    owned_products = set()

    universal_llm_chat_url = f"{universal_api['url'].rstrip('/')}/chat/completions?api-version={inference_api_version}"
    unified_ai_v1_chat_url = f"{unified_api['url'].rstrip('/')}/v1/chat/completions"


### 3️⃣ Acquire JWT Token

Setup grants Model.Invoke directly to a dedicated validation client's service principal. A second client has MCP.Invoke but not Model.Invoke. Acquire fresh same-audience tokens for both from setup's saved credentials.

Inspect or exercise 3️⃣ Acquire JWT Token using the configured identity fixture, without displaying or saving credential bytes.

**Input:** `entra_client_id`, `entra_audience`

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [4]:
with utils.notebook_step('3️⃣ Acquire Application JWT Tokens', inputs={'client_id': entra_client_id}, expected='Fresh role-bearing and missing-model-role application tokens in memory.', namespace=globals()):
    jwt_token = acquire_application_token(entra_tenant_id, entra_audience,
        entra_client_id, entra_client_secret)
    no_role_token_fixture = jwt_settings["no_role_token"] or acquire_application_token(
        entra_tenant_id, entra_audience, jwt_settings["negative_client_id"],
        jwt_settings["negative_client_secret"])
    utils.print_ok("Application tokens acquired; credentials and token bytes are not displayed.")


### 3️⃣.b Acquire Current Signed-In User JWT

Acquire the current Azure CLI user's delegated token, or use an explicit override, without displaying or saving token bytes.

**Input:** Current Azure CLI user session and setup's tenant/audience; optional delegated-token override.

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [5]:
with utils.notebook_step('3️⃣.b Acquire Current User JWT', inputs={'tenant': entra_tenant_id}, expected='A delegated gateway token for the current Azure CLI user, without a sign-in prompt.', namespace=globals()):
    current_user_id = signed_in_user(context.account)
    device_flow_token = device_token_fixture or acquire_user_token(entra_tenant_id, entra_audience)
    utils.print_ok("Delegated token acquired for the current Azure CLI user.")


### 3️⃣.c Inspect Issued JWT Tokens

Decode headers and selected claims locally without printing token bytes. Check
signing algorithm/key ID, version, audience, issuer, tenant, lifetime, client/user
identity, delegated scope, and scenario-specific required/forbidden roles.

Decoding is not signature verification. Live APIM tests provide that evidence,
including the JWT-only control before the missing-Model.Invoke denial.

Inspect or exercise 3️⃣.c Inspect Issued JWT Tokens using the configured identity fixture, without displaying or saving credential bytes.

**Input:** `roles`, `parts`, `payload`

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [6]:
with utils.notebook_step('3️⃣.c Inspect Issued JWT Tokens', inputs={}, expected='Decoded header/claims and explicit checks for each positive and negative identity.', namespace=globals()):
    token_inspections = {}
    for label, token, expectations in [
        ("service-principal", jwt_token, {"client_id": entra_client_id, "required_role": "Model.Invoke"}),
        ("current-user", device_flow_token, {"delegated": True, "user_id": current_user_id, "required_role": "Model.Invoke"}),
        ("missing-model-role", no_role_token_fixture, {
            "client_id": jwt_settings["negative_client_id"], "forbidden_role": "Model.Invoke"}),
    ]:
        # Explicit negative-token overrides must still match the expected issuer and audience.
        if label == "missing-model-role" and jwt_settings["no_role_token"]:
            from jwttools import decode_jwt
            expectations["client_id"] = decode_jwt(token)[1].get("azp", "")
        report = validate_jwt(token, tenant_id=entra_tenant_id,
            resource_id=jwt_settings["resource_id"], **expectations)
        token_inspections[label] = report
        utils.format_output(f"Decoded JWT: {label}", report)
        run.check(f"claims/{label}", True, expected="Correct identity, issuer, audience, lifetime and scenario roles", actual="PASS")
    utils.print_info("Decoding does not verify signatures. The APIM success/control calls prove cryptographic acceptance.")


### 3️⃣.d Select Active Token for Testing

Inspect or exercise 3️⃣.d Select Active Token for Testing using the configured identity fixture, without displaying or saving credential bytes.

**Input:** `parts`, `roles`

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [7]:
with utils.notebook_step('3️⃣.d Select Active Token for Testing', inputs={'token_source': token_source}, expected='Both positive identities will be exercised; active_token remains available for manual inspection.', namespace=globals()):
    positive_tokens = {"service-principal": jwt_token, "current-user": device_flow_token}
    active_token = device_flow_token if token_source in ("current_user", "device_flow") else jwt_token
    utils.require_inputs({"jwt.active_token": active_token})
    utils.print_info("Positive tests cover both the service principal and the current signed-in user.")


---
<a id='use-case-1'></a>
## 🛡️ Use Case 1: JWT-Enforced Access Contract

This use case creates an access contract that requires **both API key and JWT token** for all requests.
The `jwtRequired` variable is set to `"true"` in the product policy, which triggers the `security-handler`
fragment to enforce JWT validation across all 3 API endpoints.

---

### 4️⃣.1 Define JWT Access Contract Configuration

Inspect or exercise 4️⃣.1 Define JWT Access Contract Configuration using the configured identity fixture, without displaying or saving credential bytes.

**Input:** `run_id`

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [8]:
with utils.notebook_step('4️⃣.1 Define JWT Access Contract Configuration', inputs={'run_id': globals().get('run_id')}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    timestamp = run_id

    # JWT-Enforced Access Contract Configuration
    jwt_contract = {
        "name": f"jwt-auth-contract-{timestamp}",
        "business_unit": "Security",
        "use_case_name": "JwtAuth-" + run_id,
        "environment": "DEV",
        "use_keyvault": use_keyvault_integration,
        "endpoint_secret": "SEC-JWT-LLM-ENDPOINT",
        "apikey_secret": "SEC-JWT-LLM-KEY",
        "description": "Security JWT Auth - Enforced Valid JWT + API Key across all endpoints"
    }

    jwt_product_id = f"LLM-{jwt_contract['business_unit']}-{jwt_contract['use_case_name']}-{jwt_contract['environment']}"

    utils.print_info(f"JWT Access Contract Configuration:")
    utils.print_info(f"  Business Unit: {jwt_contract['business_unit']}")
    utils.print_info(f"  Use Case: {jwt_contract['use_case_name']}")
    utils.print_info(f"  JWT Enforcement: ENABLED")
    utils.print_info(f"  Product ID: {jwt_product_id}")


### 4️⃣.2 Create JWT Product Policy

Inspect or exercise 4️⃣.2 Create JWT Product Policy using the configured identity fixture, without displaying or saving credential bytes.

**Input:** `model_name`

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [9]:
with utils.notebook_step('4️⃣.2 Create JWT Product Policy', inputs={'model_name': globals().get('model_name')}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    jwt_product_policy = access_policy([model_name], 5000, 100000, "Monthly", jwt_required=True)
    utils.format_output('4️⃣.2 Create JWT Product Policy', {'jwt_product_policy': jwt_product_policy}, status='info', part='RESULT')


### 4️⃣.3 Create Parameter File and Deploy JWT Access Contract

Generate the editable parameter artifact for 4️⃣.3 Create Parameter File and Deploy JWT Access Contract and display the proposed configuration without deploying it.

**Input:** `jwt_contract`, `jwt_product_policy`, `run_dir`

**Output:** A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.


In [10]:
with utils.notebook_step('4️⃣.3 Create Parameter File and Deploy JWT Access Contract', inputs={'jwt_contract': globals().get('jwt_contract'), 'jwt_product_policy': globals().get('jwt_product_policy'), 'run_dir': globals().get('run_dir')}, expected='A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.', namespace=globals()):
    jwt_parameters = access_parameters(context, apimClientTool, jwt_contract, jwt_product_policy,
        api_names=[universal_api["name"], unified_api["name"]],
        key_vault={"subscriptionId": keyvault_subscription_id, "resourceGroupName": keyvault_resource_group, "name": keyvault_name} if use_keyvault_integration else None)

    jwt_params_file = write_parameters(
        repo_root / "bicep" / "infra" / "citadel-access-contracts" / "main.bicep",
        jwt_parameters, run_dir / "jwt")
    utils.format_output('4️⃣.3 Create Parameter File and Deploy JWT Access Contract', {'value_1': preview(jwt_params_file)}, status='info', part='RESULT')


### 4️⃣.4 Retrieve API Key for JWT Access Contract

Deploy the reviewed artifact for 4️⃣.4 Retrieve API Key for JWT Access Contract, then inspect its real resource outputs and ownership.

**Input:** `jwt_params_file`, `location`, `preexisting_products`, `owned_products`, `jwt_contract`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [11]:
with utils.notebook_step('4️⃣.4 Retrieve API Key for JWT Access Contract', inputs={'jwt_params_file': globals().get('jwt_params_file'), 'location': globals().get('location'), 'preexisting_products': globals().get('preexisting_products'), 'owned_products': globals().get('owned_products'), 'jwt_contract': globals().get('jwt_contract')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    jwt_outputs = deploy_contract(jwt_params_file, location, jwt_contract["name"],
        required_outputs=("products", "subscriptions"))
    jwt_product_id, jwt_subscription_name = contract_identity(
        jwt_outputs, preexisting_products, owned_products)
    jwt_api_key = apimClientTool.get_subscription_key(jwt_subscription_name)


---
## 🧪 Test Suite: JWT Authentication Across Universal and Unified OpenAI v1 APIs

The following tests validate the unified `security-handler` behavior:
1. **API Key + JWT** on each endpoint (should succeed)
2. **API Key only** on each endpoint (should fail with 401 - JWT required)
3. **JWT only** without API key (should fail with 401 - API key required)
4. **Invalid JWT** with valid API key (should fail with 401 - invalid token)

---

### 5️⃣.1 Define Test Endpoints and Payloads

Exercise 5️⃣.1 Define Test Endpoints and Payloads and compare the actual response/evidence with the expected feature behavior.

**Input:** `universal_llm_chat_url`, `unified_ai_v1_chat_url`, `model_name`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [12]:
with utils.notebook_step('5️⃣.1 Define Test Endpoints and Payloads', inputs={'universal_llm_chat_url': globals().get('universal_llm_chat_url'), 'unified_ai_v1_chat_url': globals().get('unified_ai_v1_chat_url'), 'model_name': globals().get('model_name')}, expected='Run authentication and role checks on Universal and Unified OpenAI v1 only.', namespace=globals()):
    api_endpoints = [
        {
            "name": "Universal LLM API",
            "url": universal_llm_chat_url,
            "payload": {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": "You are a helpful assistant. Be very concise."},
                    {"role": "user", "content": "Say hello in exactly 3 words."}
                ],
                "max_tokens": 20,
                "temperature": 0.3
            }
        },
        {
            "name": "Unified AI API (OpenAI v1)",
            "url": unified_ai_v1_chat_url,
            "payload": {
                "messages": [
                    {"role": "system", "content": "You are a helpful assistant. Be very concise."},
                    {"role": "user", "content": "Say hello in exactly 3 words."}
                ],
                "max_tokens": 20,
                "temperature": 0.3,
                "model": model_name
            }
        }
    ]

    utils.print_info(f"Defined {len(api_endpoints)} API endpoints for testing:")
    for ep in api_endpoints:
        utils.print_info(f"  - {ep['name']}: {ep['url'][:80]}...")

    def token_claims(token):
        if not token or len(token.split(".")) != 3:
            return {}
        part = token.split(".")[1]
        try:
            claims = json.loads(base64.urlsafe_b64decode(part + "=" * (-len(part) % 4)))
            return claims if isinstance(claims, dict) else {}
        except (ValueError, TypeError):
            return {}

    def auth_case(case, api_key_value, token, expected_status, error_code=None, required_role=None):
        for ep in api_endpoints:
            case_id = f"{case}/{ep['name']}"
            if (expected_status == 200 or case == "04-jwt-without-key") and not token:
                run.blocked(case_id, "Acquire a valid client-credentials or delegated token first.")
                continue
            if required_role and required_role not in token_claims(token).get("roles", []):
                run.blocked(case_id, f"Assign {required_role} and acquire a fresh token.")
                continue
            headers = {"Content-Type": "application/json"}
            if api_key_value:
                headers["api-key"] = api_key_value
            if token:
                headers["Authorization"] = f"Bearer {token}"
            response = transport.request("POST", ep["url"], headers=headers, json=ep["payload"])
            data = response.json_body if isinstance(response.json_body, dict) else {}
            err = data.get("error", {})
            observed_code = err.get("code", "") if isinstance(err, dict) else ""
            if expected_status == 200:
                body_ok = bool(data.get("choices", [{}])[0].get("message", {}).get("content"))
            else:
                # JWT validation errors may be APIM's JSON {statusCode,message} shape.
                body_ok = observed_code == error_code if error_code else bool(
                    observed_code or (data.get("statusCode") == expected_status and data.get("message")))
            run.check(case_id, response.status_code == expected_status and body_ok and not response.error,
                expected=f"HTTP {expected_status}; {error_code or 'valid body/policy error'}",
                actual=response.status_code, api=ep["name"], model=model_name,
                error_code=observed_code, transport_error=response.error, duration=response.duration)


### 5️⃣.2 Test 1: API Key + JWT Token (Should Succeed ✅)

Inspect or exercise 5️⃣.2 Test 1: API Key + JWT Token (Should Succeed ✅) using the configured identity fixture, without displaying or saving credential bytes.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [13]:
with utils.notebook_step('5️⃣.2 Test 1: API Key + JWT Token (Should Succeed ✅)', inputs={}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    for identity, token in positive_tokens.items():
        auth_case(f"01-valid-jwt/{identity}", jwt_api_key, token, 200)


### 5️⃣.3 Test 2: API Key Only - No JWT (Should Fail ❌)

Inspect or exercise 5️⃣.3 Test 2: API Key Only - No JWT (Should Fail ❌) using the configured identity fixture, without displaying or saving credential bytes.

Send requests with only the API key (no JWT token). Since the access contract has `jwtRequired=true`,
these should be rejected with HTTP 401.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [14]:
with utils.notebook_step('5️⃣.3 Test 2: API Key Only - No JWT (Should Fail ❌)', inputs={}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    auth_case("02-key-without-jwt", jwt_api_key, None, 401, "jwt_required")


### 5️⃣.4 Test 3: Invalid JWT Token (Should Fail ❌)

Inspect or exercise 5️⃣.4 Test 3: Invalid JWT Token (Should Fail ❌) using the configured identity fixture, without displaying or saving credential bytes.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [15]:
with utils.notebook_step('5️⃣.4 Test 3: Invalid JWT Token (Should Fail ❌)', inputs={}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    auth_case("03-invalid-jwt", jwt_api_key, "INVALID-TEST-TOKEN-FOR-VALIDATION", 401)


### 5️⃣.5 Test 4: JWT Only - No API Key (Should Fail ❌)

Inspect or exercise 5️⃣.5 Test 4: JWT Only - No API Key (Should Fail ❌) using the configured identity fixture, without displaying or saving credential bytes.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [16]:
with utils.notebook_step('5️⃣.5 Test 4: JWT Only - No API Key (Should Fail ❌)', inputs={}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    auth_case("04-jwt-without-key", None, active_token, 401)


---
<a id='use-case-2'></a>
## 🛡️ Use Case 2: Role-Enforced Access Contract

> 🔄 [Back to Select Active Token](#3d) — switch between device flow and client credentials before running tests.

This use case creates an access contract that requires **both JWT token and a specific app role**.
The `requiredRoles` variable is set in the product policy, which triggers the `security-handler`
fragment to verify the token's `roles` claim contains at least one of the required roles.

**Setup prerequisites:** Model.Invoke is assigned directly to the dedicated validation client and current signed-in user. Both positive identities are tested. Service principals and managed identities do not inherit app roles from group membership.


### 6️⃣.1 Define Role-Enforced Access Contract Configuration

Prepare the 6️⃣.1 Define Role-Enforced Access Contract Configuration state shown below for its subsequent feature-specific requests and assertions.

**Input:** `run_id`, `timestamp`

**Output:** `role_contract`, `role_product_id` are available to the following cells.


In [17]:
with utils.notebook_step('6️⃣.1 Define Role-Enforced Access Contract Configuration', inputs={'run_id': globals().get('run_id'), 'timestamp': globals().get('timestamp')}, expected='`role_contract`, `role_product_id` are available to the following cells.', namespace=globals()):
    # Role-Enforced Access Contract Configuration
    role_contract = {
        "name": f"role-auth-contract-{timestamp}",
        "business_unit": "Security",
        "use_case_name": "RoleAuth-" + run_id,
        "environment": "DEV",
        "use_keyvault": use_keyvault_integration,
        "endpoint_secret": "SEC-ROLE-LLM-ENDPOINT",
        "apikey_secret": "SEC-ROLE-LLM-KEY",
        "description": "Security Role Auth - JWT + App Role enforcement across all endpoints"
    }

    role_product_id = f"LLM-{role_contract['business_unit']}-{role_contract['use_case_name']}-{role_contract['environment']}"

    utils.print_info(f"Role Access Contract Configuration:")
    utils.print_info(f"  Business Unit: {role_contract['business_unit']}")
    utils.print_info(f"  Use Case: {role_contract['use_case_name']}")
    utils.print_info(f"  JWT Enforcement: ENABLED")
    utils.print_info(f"  Required Role: Model.Invoke")
    utils.print_info(f"  Product ID: {role_product_id}")


### 6️⃣.2 Create Role-Enforced Product Policy and Deploy

Generate the editable parameter artifact for 6️⃣.2 Create Role-Enforced Product Policy and Deploy and display the proposed configuration without deploying it.

**Input:** `role_contract`, `model_name`, `run_dir`

**Output:** A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.


In [18]:
with utils.notebook_step('6️⃣.2 Create Role-Enforced Product Policy and Deploy', inputs={'role_contract': globals().get('role_contract'), 'model_name': globals().get('model_name'), 'run_dir': globals().get('run_dir')}, expected='A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.', namespace=globals()):
    role_product_policy = access_policy([model_name], 5000, 100000, "Monthly",
        jwt_required=True, inbound_extra='<set-variable name="requiredRoles" value="Model.Invoke" />')
    utils.format_output('6️⃣.2 Create Role-Enforced Product Policy and Deploy', {'role_product_policy': role_product_policy}, status='info', part='RESULT')

    role_parameters = access_parameters(context, apimClientTool, role_contract, role_product_policy,
        api_names=[universal_api["name"], unified_api["name"]],
        key_vault={"subscriptionId": keyvault_subscription_id, "resourceGroupName": keyvault_resource_group, "name": keyvault_name} if use_keyvault_integration else None)

    role_params_file = write_parameters(
        repo_root / "bicep" / "infra" / "citadel-access-contracts" / "main.bicep",
        role_parameters, run_dir / "role")
    utils.format_output('6️⃣.2 Create Role-Enforced Product Policy and Deploy', {'value_1': preview(role_params_file)}, status='info', part='RESULT')


### 6️⃣.3 Retrieve API Key for Role-Enforced Access Contract

Deploy the reviewed artifact for 6️⃣.3 Retrieve API Key for Role-Enforced Access Contract, then inspect its real resource outputs and ownership.

**Input:** `role_params_file`, `location`, `preexisting_products`, `owned_products`, `role_contract`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [19]:
with utils.notebook_step('6️⃣.3 Retrieve API Key for Role-Enforced Access Contract', inputs={'role_params_file': globals().get('role_params_file'), 'location': globals().get('location'), 'preexisting_products': globals().get('preexisting_products'), 'owned_products': globals().get('owned_products'), 'role_contract': globals().get('role_contract')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    role_outputs = deploy_contract(role_params_file, location, role_contract["name"],
        required_outputs=("products", "subscriptions"))
    role_product_id, role_subscription_name = contract_identity(
        role_outputs, preexisting_products, owned_products)
    role_api_key = apimClientTool.get_subscription_key(role_subscription_name)


### 7️⃣.1 Test 5: API Key + JWT with Correct Role (Should Succeed ✅)

Inspect or exercise 7️⃣.1 Test 5: API Key + JWT with Correct Role (Should Succeed ✅) using the configured identity fixture, without displaying or saving credential bytes.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [20]:
with utils.notebook_step('7️⃣.1 Test 5: JWT with Correct Role', inputs={}, expected='Both identities succeed with Model.Invoke on every API surface.', namespace=globals()):
    for identity, token in positive_tokens.items():
        auth_case(f"05-role-bearing-jwt/{identity}", role_api_key, token, 200, required_role="Model.Invoke")


### 7️⃣.2 Test 6: Genuine JWT Without Required Role (403 insufficientrole)

Inspect or exercise 7️⃣.2 Test 6: Genuine JWT Without Required Role (403 insufficientrole) using the configured identity fixture, without displaying or saving credential bytes.

**Input:** `api_endpoints`, `headers`, `case_id`, `ep`

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [21]:
with utils.notebook_step('7️⃣.2 Test 6: Genuine JWT Without Required Role (403 insufficientrole)', inputs={'api_endpoints': globals().get('api_endpoints'), 'headers': globals().get('headers'), 'case_id': globals().get('case_id'), 'ep': globals().get('ep')}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    # Prove validity using the JWT-only product before testing the role-enforced product.
    no_role_token = no_role_token_fixture or (
        active_token if "Model.Invoke" not in token_claims(active_token).get("roles", []) else "")
    for ep in api_endpoints:
        case_id = f"06-missing-role/{ep['name']}"
        claims = token_claims(no_role_token)
        if (not no_role_token or "Model.Invoke" in claims.get("roles", [])
            or not isinstance(claims.get("exp"), (int, float)) or claims["exp"] <= time.time()):
            run.blocked(case_id, "The negative client token must be fresh, issued for this gateway, and lack Model.Invoke. Rerun setup/token acquisition or provide a valid override. No invalid-token fallback.")
            continue
        headers = {"api-key": jwt_api_key, "Authorization": f"Bearer {no_role_token}"}
        control = transport.request("POST", ep["url"], headers=headers, json=ep["payload"])
        control_data = control.json_body if isinstance(control.json_body, dict) else {}
        if control.status_code != 200 or not control_data.get("choices", [{}])[0].get("message", {}).get("content"):
            run.blocked(case_id, "No-role token failed the non-role JWT product control; role rejection is not proven.")
            continue
        headers["api-key"] = role_api_key
        denied = transport.request("POST", ep["url"], headers=headers, json=ep["payload"])
        data = denied.json_body if isinstance(denied.json_body, dict) else {}
        run.check(case_id, denied.status_code == 403 and data.get("error", {}).get("code") == "insufficient_role",
            expected="Valid token accepted by JWT-only product, then 403 insufficient_role",
            actual=denied.status_code, control_status=control.status_code, api=ep["name"])


### 7️⃣.3 Test 7: API Key Only on Role-Enforced Product (Should Fail ❌)

Prepare the 7️⃣.3 Test 7: API Key Only on Role-Enforced Product (Should Fail ❌) state shown below for its subsequent feature-specific requests and assertions.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** The visible 7️⃣.3 Test 7: API Key Only on Role-Enforced Product (Should Fail ❌) inspection output.


In [22]:
with utils.notebook_step('7️⃣.3 Test 7: API Key Only on Role-Enforced Product (Should Fail ❌)', inputs={}, expected='The visible 7️⃣.3 Test 7: API Key Only on Role-Enforced Product (Should Fail ❌) inspection output.', namespace=globals()):
    auth_case("07-role-product-no-jwt", role_api_key, None, 401, "jwt_required")


---
## 📊 Results Summary
---

### 📊 Results Summary

Collect the final 📊 Results Summary verdict without discarding earlier failed or blocked required cases.

**Input:** The initialized configuration and the explicit literals in this cell.

**Output:** The strict validation summary; any failed or blocked required coverage raises.


In [23]:
with utils.notebook_step('📊 Results Summary', inputs={}, expected='The strict validation summary; any failed or blocked required coverage raises.', namespace=globals()):
    run.finish()


case,status,required,expected,actual
claims/service-principal,PASS,True,"Correct identity, issuer, audience, lifetime and scenario roles",PASS
claims/current-user,PASS,True,"Correct identity, issuer, audience, lifetime and scenario roles",PASS
claims/missing-model-role,PASS,True,"Correct identity, issuer, audience, lifetime and scenario roles",PASS
01-valid-jwt/service-principal/Universal LLM API,PASS,True,HTTP 200; valid body/policy error,200
01-valid-jwt/service-principal/Unified AI API (OpenAI v1),PASS,True,HTTP 200; valid body/policy error,200
01-valid-jwt/current-user/Universal LLM API,PASS,True,HTTP 200; valid body/policy error,200
01-valid-jwt/current-user/Unified AI API (OpenAI v1),PASS,True,HTTP 200; valid body/policy error,200
02-key-without-jwt/Universal LLM API,PASS,True,HTTP 401; jwt_required,401
02-key-without-jwt/Unified AI API (OpenAI v1),PASS,True,HTTP 401; jwt_required,401
03-invalid-jwt/Universal LLM API,PASS,True,HTTP 401; valid body/policy error,401


<a id='cleanup'></a>
### Cleanup (Optional)

Remove the test JWT access contract from APIM created during this notebook session.

> **Note:** This will not delete any Entra ID app registrations or APIM named values.

### Cleanup (Optional)

Apply the explicitly enabled Cleanup (Optional) action only to resources owned by this run.

**Input:** `owned_products`, `cleanup_delete_contract`

**Output:** Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.


In [24]:
with utils.notebook_step('Cleanup (Optional)', inputs={'owned_products': globals().get('owned_products'), 'cleanup_delete_contract': globals().get('cleanup_delete_contract')}, expected='Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.', namespace=globals()):
    cleanup_products(apimClientTool, owned_products, enabled=cleanup_delete_contract)


Current run's owned products: ['LLM-Security-JwtAuth-907ea034f3d1-DEV', 'LLM-Security-RoleAuth-907ea034f3d1-DEV']; cleanup enabled: False
